# Generative Recipe Model — Train on Google Colab

This notebook trains a language model that **invents brand-new recipes** (not
retrieval — actual free-text generation), conditioned on a mood/craving like
*"something sweet"* or *"I want something spicy and healthy"*.

This is a **different component** from your existing `ml_engine` /
`loseit_project/backend` recommender (`train.py`, which scores/ranks the
existing 300 recipes for a user). That system picks the best *existing*
recipe. This notebook builds a **generator**: it can produce recipes that
were never in any dataset. The two are complementary — recommend from the
catalog when a good match exists, generate something new when the user wants
novelty or the catalog has a gap.

## Approach: two-stage fine-tuning

1. **Stage 1 — learn "how to write a recipe"**: fine-tune GPT-2 (`distilgpt2`)
   on the public **Food.com** dataset (~230k recipes with full ingredient
   lists and step-by-step instructions). This teaches the model recipe
   *structure* — how ingredients lists look, how steps are phrased, how
   length and formatting work.
2. **Stage 2 — specialise toward Sri Lankan / weight-loss recipes**: continue
   fine-tuning on your own `Sri_Lankan_Recipe_Database (1).xlsx` (300
   recipes), mixed with a small replay sample of Food.com examples. This
   shifts the model's vocabulary and ingredient choices toward Sri Lankan
   cooking and nutrition-aware tags, **without erasing** what it learned
   about writing full instructions in Stage 1 (a known failure mode called
   *catastrophic forgetting* — the replay sample is what prevents it).

**Important limitation to note in your report:** your Sri Lankan dataset has
ingredients and nutrition but **no step-by-step instructions**. So for
Sri-Lankan-flavoured generations, the *steps text* is stylistically inferred
from the (larger) Food.com corpus, not from real Sri Lankan cooking
instructions. The dish names, ingredients, and nutrition profile will be
authentically Sri Lankan; the exact phrasing of steps should be treated as a
draft to be reviewed, not a verified recipe. This is worth stating explicitly
as a limitation/future-work item — e.g. "future work: collect real
instruction text for the 300-recipe set to remove this gap."

## Runtime
Runtime → Change runtime type → **T4 GPU** (or better) before running.


## 0. Install dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate kaggle openpyxl
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


## 1. Stage 1 — download the Food.com dataset

We'll use the **Food.com Recipes and Interactions** dataset on Kaggle
(`shuyangli94/food-com-recipes-and-user-interactions`), ~230k recipes.

You need a free Kaggle API token:
1. Go to kaggle.com → your profile picture → **Settings** → **API** → **Create New Token**.
2. This downloads a `kaggle.json` file.
3. Run the cell below and upload that file when prompted.


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload kaggle.json here

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d shuyangli94/food-com-recipes-and-user-interactions -f RAW_recipes.csv -p /content/foodcom --force
!unzip -o /content/foodcom/RAW_recipes.csv.zip -d /content/foodcom
!ls -la /content/foodcom


## 2. Load and inspect Food.com data

In [ ]:
import pandas as pd
import ast

foodcom_df = pd.read_csv("/content/foodcom/RAW_recipes.csv")
print(foodcom_df.shape)
foodcom_df.head(3)


## 3. Curate a "mood" tag vocabulary

`tags` is a stringified Python list per row (e.g. `"['60-minutes-or-less', 'desserts', ...]"`).
We check which taste/mood-relevant tags actually occur often enough in the
data to be useful conditioning signals — this is data-driven, not guessed,
so it won't reference tags that don't exist in this dataset.


In [ ]:
from collections import Counter

foodcom_df["tags_list"] = foodcom_df["tags"].apply(ast.literal_eval)

CANDIDATE_MOOD_TAGS = [
    "sweet", "desserts", "spicy", "savory", "salty", "sour", "healthy",
    "low-calorie", "comfort-food", "quick", "easy", "vegetarian", "vegan",
    "breakfast", "lunch", "dinner", "snacks", "soups-stews", "seafood",
    "chicken", "beef", "pork", "chocolate", "fruit", "summer", "winter",
    "holiday-event", "kid-friendly", "main-dish", "side-dishes",
    "appetizers", "beverages", "spicy",
]

tag_counts = Counter()
for tags in foodcom_df["tags_list"]:
    tag_counts.update(tags)

KEPT_TAGS = sorted({t for t in CANDIDATE_MOOD_TAGS if tag_counts.get(t, 0) > 200})
print(f"Kept {len(KEPT_TAGS)} mood/taste tags (found in dataset with count > 200):")
for t in KEPT_TAGS:
    print(f"  {t}: {tag_counts[t]}")


## 4. Build the training text format

Every recipe becomes one plain-text example with a fixed schema. The special
markers `<|startofrecipe|>` / `<|endofrecipe|>` let the model learn where a
recipe begins and ends, so at generation time we can stop cleanly.

```
<|startofrecipe|>
tags: sweet, desserts, easy
title: Chocolate Lava Cake
ingredients:
- 200 g dark chocolate
- 3 eggs
...
steps:
1. Preheat oven to 200C.
2. Melt chocolate ...
<|endofrecipe|>
```


In [ ]:
import ast

foodcom_df["ingredients_list"] = foodcom_df["ingredients"].apply(ast.literal_eval)
foodcom_df["steps_list"] = foodcom_df["steps"].apply(ast.literal_eval)

def format_foodcom_row(row):
    tags = [t for t in row["tags_list"] if t in KEPT_TAGS][:6]
    lines = ["<|startofrecipe|>"]
    lines.append("tags: " + (", ".join(tags) if tags else "general"))
    lines.append("title: " + str(row["name"]).strip())
    lines.append("ingredients:")
    for ing in row["ingredients_list"]:
        lines.append("- " + str(ing).strip())
    lines.append("steps:")
    for i, step in enumerate(row["steps_list"], 1):
        lines.append(f"{i}. " + str(step).strip())
    lines.append("<|endofrecipe|>")
    return "\n".join(lines)

# Quality filter: keep recipes with a sane number of ingredients/steps and a title
mask = (
    foodcom_df["n_ingredients"].between(2, 20)
    & foodcom_df["n_steps"].between(2, 15)
    & foodcom_df["name"].str.len().fillna(0).gt(2)
)
clean_df = foodcom_df[mask].reset_index(drop=True)
print(f"Kept {len(clean_df)} / {len(foodcom_df)} recipes after filtering")

# Subsample to keep Stage-1 training time reasonable on a free Colab GPU.
# Raise this if you have more time/compute.
SAMPLE_SIZE = 60000
if len(clean_df) > SAMPLE_SIZE:
    clean_df = clean_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

clean_df["formatted_text"] = clean_df.apply(format_foodcom_row, axis=1)
print(clean_df["formatted_text"].iloc[0])


## 5. Train/validation split

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts = train_test_split(
    clean_df["formatted_text"].tolist(), test_size=0.02, random_state=42
)
print(f"Train: {len(train_texts)}  Val: {len(val_texts)}")


## 6. Tokenizer setup

In [ ]:
from transformers import GPT2Tokenizer

BASE_MODEL = "distilgpt2"  # swap for "gpt2" or "gpt2-medium" if you have more GPU time

tokenizer = GPT2Tokenizer.from_pretrained(BASE_MODEL)
tokenizer.add_special_tokens({
    "pad_token": "<|pad|>",
    "additional_special_tokens": ["<|startofrecipe|>", "<|endofrecipe|>"],
})
print("Vocab size:", len(tokenizer))


## 7. Tokenize and chunk into fixed-size blocks

Standard causal-LM fine-tuning approach: tokenize every example, concatenate
all the token ids for a split, then slice into fixed-length blocks. This
avoids wasting compute on padding.


In [ ]:
from datasets import Dataset

BLOCK_SIZE = 256

def tokenize_and_group(texts, batch_size=1000):
    all_ids = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, add_special_tokens=False)
        for ids in enc["input_ids"]:
            all_ids.extend(ids)
    total_len = (len(all_ids) // BLOCK_SIZE) * BLOCK_SIZE
    blocks = [all_ids[i:i + BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
    return Dataset.from_dict({
        "input_ids": blocks,
        "labels": [b.copy() for b in blocks],
    })

train_dataset_stage1 = tokenize_and_group(train_texts)
val_dataset_stage1 = tokenize_and_group(val_texts)
print(train_dataset_stage1)
print(val_dataset_stage1)


## 8. Load model and train — Stage 1 (Food.com)

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer))
model.to("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from transformers import TrainingArguments, Trainer

stage1_args = TrainingArguments(
    output_dir="/content/stage1_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    learning_rate=5e-5,
    warmup_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_stage1 = Trainer(
    model=model,
    args=stage1_args,
    train_dataset=train_dataset_stage1,
    eval_dataset=val_dataset_stage1,
)

trainer_stage1.train()


In [ ]:
import math

eval_results = trainer_stage1.evaluate()
print(f"Stage 1 eval loss: {eval_results['eval_loss']:.4f}  perplexity: {math.exp(eval_results['eval_loss']):.2f}")

STAGE1_DIR = "/content/recipe_gpt2_stage1_foodcom"
trainer_stage1.save_model(STAGE1_DIR)
tokenizer.save_pretrained(STAGE1_DIR)
print("Saved Stage 1 model to", STAGE1_DIR)


### Sanity check — generate from the Stage 1 model before continuing

In [ ]:
def quick_generate(prompt, model, tokenizer, max_new_tokens=200):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_p=0.92,
        top_k=50,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=False)

print(quick_generate("<|startofrecipe|>\ntags: sweet, desserts\ntitle:", model, tokenizer))


## 9. Stage 2 — transfer to your Sri Lankan recipe dataset

Upload `Sri_Lankan_Recipe_Database (1).xlsx` from
`Project Files/Sri_Lankan_Recipe_Database (1).xlsx` when prompted.


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload "Sri_Lankan_Recipe_Database (1).xlsx"

import glob
xlsx_path = glob.glob("*.xlsx")[0]
print("Using file:", xlsx_path)


In [ ]:
sl_df = pd.read_excel(xlsx_path, sheet_name="Recipes")
print(sl_df.shape)
sl_df.head(3)


### Preprocess Sri Lankan recipes

`Weight_Loss_Suitability` in the spreadsheet is an Excel formula
(`=IF(calories<200,"Low-Cal",...)`); we recompute it directly from
`Calories_kcal` in pandas rather than trusting a cached formula value, since
that's more robust to how the file gets re-saved.

Note: this dataset has **no instructions/steps column** — only
`Key_Ingredients` and a one-line `Short_Description`. We intentionally leave
"steps:" out of these training examples (see the limitation noted at the top
of this notebook); Stage 2 still includes replayed Food.com examples so the
model doesn't lose its ability to write a steps section.


In [ ]:
def weight_loss_suitability(cal):
    if cal < 200:
        return "Low-Cal"
    elif cal < 400:
        return "Moderate"
    else:
        return "High-Cal"

sl_df["Weight_Loss_Suitability"] = sl_df["Calories_kcal"].apply(weight_loss_suitability)

def format_srilankan_row(row):
    tag_fields = ["Category", "Food_Group", "Dietary_Tag", "Meal_Type", "Spice_Level", "Weight_Loss_Suitability"]
    tags = []
    for field in tag_fields:
        v = str(row[field]).strip().lower()
        if v and v != "nan" and v not in tags:
            tags.append(v)

    ingredients = [x.strip() for x in str(row["Key_Ingredients"]).split(",") if x.strip()]

    lines = ["<|startofrecipe|>"]
    lines.append("tags: " + ", ".join(tags))
    lines.append("title: " + str(row["Recipe_Name"]).strip())
    lines.append("ingredients:")
    for ing in ingredients:
        lines.append("- " + ing)
    lines.append(
        "nutrition: {:.0f} kcal, {:.1f}g protein, {:.1f}g carbs, {:.1f}g fat, {:.1f}g fiber".format(
            row["Calories_kcal"], row["Protein_g"], row["Carbs_g"], row["Fat_g"], row["Fiber_g"]
        )
    )
    desc = str(row["Short_Description"]).strip()
    if desc and desc.lower() != "nan":
        lines.append("description: " + desc)
    lines.append("<|endofrecipe|>")
    return "\n".join(lines)

sl_df["formatted_text"] = sl_df.apply(format_srilankan_row, axis=1)
print(sl_df["formatted_text"].iloc[0])


### Build the Stage 2 training set: oversampled Sri Lankan data + Food.com replay

300 recipes is too little to fine-tune on alone (the model would overfit /
forget general recipe-writing ability). We:
- repeat the Sri Lankan examples a few times (with tag order shuffled each
  time, for mild variation) so they carry meaningful weight during training,
- mix in a modest random replay sample from the Stage 1 Food.com training
  texts, so instruction-writing ability is retained.


In [ ]:
import random

random.seed(42)

def shuffled_tags_variant(row):
    tag_fields = ["Category", "Food_Group", "Dietary_Tag", "Meal_Type", "Spice_Level", "Weight_Loss_Suitability"]
    tags = []
    for field in tag_fields:
        v = str(row[field]).strip().lower()
        if v and v != "nan" and v not in tags:
            tags.append(v)
    random.shuffle(tags)
    ingredients = [x.strip() for x in str(row["Key_Ingredients"]).split(",") if x.strip()]
    lines = ["<|startofrecipe|>", "tags: " + ", ".join(tags), "title: " + str(row["Recipe_Name"]).strip(), "ingredients:"]
    lines += ["- " + ing for ing in ingredients]
    lines.append(
        "nutrition: {:.0f} kcal, {:.1f}g protein, {:.1f}g carbs, {:.1f}g fat, {:.1f}g fiber".format(
            row["Calories_kcal"], row["Protein_g"], row["Carbs_g"], row["Fat_g"], row["Fiber_g"]
        )
    )
    desc = str(row["Short_Description"]).strip()
    if desc and desc.lower() != "nan":
        lines.append("description: " + desc)
    lines.append("<|endofrecipe|>")
    return "\n".join(lines)

OVERSAMPLE_FACTOR = 4
sl_texts = []
for _ in range(OVERSAMPLE_FACTOR):
    sl_texts.extend(sl_df.apply(shuffled_tags_variant, axis=1).tolist())

REPLAY_SIZE = 3000
replay_texts = random.sample(train_texts, min(REPLAY_SIZE, len(train_texts)))

stage2_texts = sl_texts + replay_texts
random.shuffle(stage2_texts)
print(f"Stage 2 training examples: {len(stage2_texts)} "
      f"({len(sl_texts)} Sri Lankan + {len(replay_texts)} Food.com replay)")

stage2_train_texts, stage2_val_texts = train_test_split(stage2_texts, test_size=0.05, random_state=42)
train_dataset_stage2 = tokenize_and_group(stage2_train_texts)
val_dataset_stage2 = tokenize_and_group(stage2_val_texts)
print(train_dataset_stage2)


### Continue training from the Stage 1 checkpoint

In [ ]:
stage2_model = GPT2LMHeadModel.from_pretrained(STAGE1_DIR)
stage2_model.to("cuda" if torch.cuda.is_available() else "cpu")

stage2_args = TrainingArguments(
    output_dir="/content/stage2_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    learning_rate=3e-5,   # lower LR for the fine-tune-on-a-fine-tune pass
    warmup_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_stage2 = Trainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=train_dataset_stage2,
    eval_dataset=val_dataset_stage2,
)

trainer_stage2.train()


In [ ]:
eval_results_2 = trainer_stage2.evaluate()
print(f"Stage 2 eval loss: {eval_results_2['eval_loss']:.4f}  perplexity: {math.exp(eval_results_2['eval_loss']):.2f}")

FINAL_DIR = "/content/recipe_gpt2_final_srilankan"
trainer_stage2.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("Saved final model to", FINAL_DIR)


## 10. Mood-conditioned generation

`generate_recipe("I'm in the mood for something sweet")` maps free-text mood
phrases to conditioning tags, builds a prompt, and samples a genuinely new
recipe — sampling (not greedy decoding) is what lets the output diverge from
anything literally in the training data.


In [ ]:
MOOD_KEYWORDS = {
    "sweet": ["sweet", "dessert"],
    "dessert": ["dessert", "sweet"],
    "spicy": ["spicy", "hot"],
    "hot": ["hot", "spicy"],
    "savory": ["savory"],
    "salty": ["salty"],
    "healthy": ["healthy", "low-cal"],
    "light": ["low-cal", "light"],
    "diet": ["low-cal"],
    "lose weight": ["low-cal"],
    "low calorie": ["low-cal"],
    "comfort": ["comfort-food"],
    "quick": ["quick", "easy"],
    "easy": ["easy"],
    "vegetarian": ["vegetarian"],
    "vegan": ["vegan"],
    "breakfast": ["breakfast"],
    "lunch": ["lunch"],
    "dinner": ["dinner"],
    "snack": ["snack", "short eats"],
    "soup": ["soup"],
    "seafood": ["pescatarian", "seafood"],
    "fish": ["pescatarian", "fish"],
    "chicken": ["chicken"],
    "chocolate": ["chocolate", "sweet"],
    "rice": ["rice & curry"],
    "curry": ["curry"],
}

def build_prompt(mood_text):
    mood_lower = mood_text.lower()
    matched = []
    for kw, tags in MOOD_KEYWORDS.items():
        if kw in mood_lower:
            for t in tags:
                if t not in matched:
                    matched.append(t)
    if not matched:
        matched = [mood_lower.strip()]
    tags_str = ", ".join(matched)
    return f"<|startofrecipe|>\ntags: {tags_str}\ntitle:"

print(build_prompt("I'm in the mood for something sweet"))
print(build_prompt("something spicy but still low calorie"))


In [ ]:
import re

def parse_generated_recipe(text):
    text = text.split("<|endofrecipe|>")[0]
    text = text.replace("<|startofrecipe|>", "").strip()

    def extract(label, next_labels):
        pattern = rf"{label}:\s*(.*?)(?=(?:{'|'.join(next_labels)}):|$)"
        m = re.search(pattern, text, re.DOTALL)
        return m.group(1).strip() if m else ""

    labels_order = ["tags", "title", "ingredients", "steps", "nutrition", "description"]
    result = {}
    for i, label in enumerate(labels_order):
        remaining = labels_order[i + 1:]
        result[label] = extract(label, remaining)
    return result

def generate_recipe(mood_text, model, tokenizer, num_return_sequences=1,
                     max_new_tokens=220, temperature=0.9, top_p=0.92, top_k=50):
    prompt = build_prompt(mood_text)
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.pad_token_id,
    )
    return [parse_generated_recipe(tokenizer.decode(o, skip_special_tokens=False)) for o in outputs]

def print_recipe(r):
    print("TITLE:", r.get("title", "").strip())
    print("TAGS:", r.get("tags", "").strip())
    print("INGREDIENTS:\n" + r.get("ingredients", "").strip())
    if r.get("steps"):
        print("STEPS:\n" + r["steps"].strip())
    if r.get("nutrition"):
        print("NUTRITION:", r["nutrition"].strip())
    if r.get("description"):
        print("DESCRIPTION:", r["description"].strip())
    print("-" * 60)


In [ ]:
for mood in ["something sweet", "spicy and healthy", "a quick vegetarian dinner"]:
    print(f"=== Mood: {mood!r} ===")
    for r in generate_recipe(mood, stage2_model, tokenizer, num_return_sequences=1):
        print_recipe(r)


## 11. Novelty check

Confirms the model isn't just reciting a training recipe: checks the
generated title for an exact match against every training title, and reports
the highest ingredient-set overlap (Jaccard similarity) against any training
recipe. Low overlap + no exact title match = a genuinely new recipe.


In [ ]:
all_training_titles = set(clean_df["name"].str.lower().str.strip()) | set(sl_df["Recipe_Name"].str.lower().str.strip())

def ingredient_set(text):
    return {line.strip("- ").lower() for line in text.split("\n") if line.strip()}

all_training_ingredient_sets = (
    [set(x) for x in clean_df["ingredients_list"].apply(lambda lst: {i.lower() for i in lst})]
    + [set(ingredient_set("\n".join(x.split(",")))) for x in sl_df["Key_Ingredients"]]
)

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def novelty_report(recipe):
    title = recipe.get("title", "").strip().lower()
    exact_title_match = title in all_training_titles
    gen_ingredients = ingredient_set(recipe.get("ingredients", ""))
    best_overlap = max((jaccard(gen_ingredients, s) for s in all_training_ingredient_sets), default=0.0)
    print(f"Title: {recipe.get('title','').strip()}")
    print(f"  Exact title seen in training data: {exact_title_match}")
    print(f"  Highest ingredient-set overlap with any training recipe: {best_overlap:.2%}")

for mood in ["something sweet", "spicy and healthy"]:
    r = generate_recipe(mood, stage2_model, tokenizer)[0]
    novelty_report(r)


## 12. Save the final model (persist beyond the Colab session)

Colab runtimes are temporary — mount Google Drive and copy the model there,
or download it as a zip, so you don't lose the trained weights when the
session disconnects.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
DRIVE_TARGET = "/content/drive/MyDrive/recipe_gpt2_final_srilankan"
shutil.copytree(FINAL_DIR, DRIVE_TARGET, dirs_exist_ok=True)
print("Copied model to Google Drive:", DRIVE_TARGET)


In [ ]:
# Alternative: download as a zip instead of using Drive
!zip -r /content/recipe_gpt2_final_srilankan.zip {FINAL_DIR}
from google.colab import files
files.download("/content/recipe_gpt2_final_srilankan.zip")


## 13. Using the model outside Colab

To use this in your `serve.py` FastAPI backend (in `loseit_project/backend`):

```python
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("recipe_gpt2_final_srilankan")
model = GPT2LMHeadModel.from_pretrained("recipe_gpt2_final_srilankan")

# then reuse build_prompt() / generate_recipe() / parse_generated_recipe()
# from this notebook as a new endpoint, e.g. POST /generate
```

## Tips for improving quality/novelty further

- **More Sri Lankan step-writing data is the single highest-value fix** —
  even 30-50 real recipes with actual instructions, added to Stage 2 with
  their own `steps:` field, would let the model learn authentic Sri Lankan
  phrasing instead of inferring it from a Western-recipe corpus.
- **Raise `SAMPLE_SIZE`** (Stage 1) and **`OVERSAMPLE_FACTOR`** (Stage 2) if
  you have more GPU time — more data generally means better generalisation.
- **Try `gpt2` or `gpt2-medium`** instead of `distilgpt2` for higher-quality
  text, at the cost of slower training/inference.
- **Tune sampling**: higher `temperature`/`top_p` → more novel but riskier
  (less coherent) output; lower → safer but closer to training patterns.
  `repetition_penalty` and `no_repeat_ngram_size` control looping/repeats.
- **For your FYP write-up**: the novelty check (Section 11) and the
  train/val perplexity numbers (Sections 8 and 9) are legitimate
  quantitative evaluation points for a "Generative Recipe Module" chapter —
  parallel to the MAE/RMSE/R² table already in your recommender's README.
